# 🎬 YouTube Trending Videos — End-to-End Data Engineering Project

> A complete data pipeline built without AWS — using free tools only.  
> Inspired by [Darshil Parmar's](https://www.youtube.com/@DarshilParmar) YouTube Data Engineering tutorial.



## 📌 Project Overview

This project builds a full data engineering pipeline that ingests, cleans, transforms, and visualizes YouTube trending video data from **10 countries**. Instead of AWS (S3, Glue, Athena, Redshift, QuickSight), this project uses **100% free alternatives**.



## 🏗️ Architecture

```
📦 Raw Data (CSV + JSON files — Kaggle)
            ↓
🔗 Google Drive (Storage — replaces AWS S3)
            ↓
🧹 Google Colab + Pandas (ETL — replaces AWS Glue)
            ↓
🗄️ Neon PostgreSQL (Data Warehouse — replaces AWS Redshift)
            ↓
📊 Looker Studio (Dashboard — replaces AWS QuickSight)
```



## 🛠️ Tech Stack (Free Alternatives to AWS)

| AWS Service | Free Alternative Used |
|-------------|----------------------|
| S3 | Google Drive |
| AWS Glue | Google Colab + Pandas |
| Athena | SQL queries via SQLAlchemy |
| Redshift | Neon PostgreSQL (neon.tech) |
| Lambda | Python scripts |
| QuickSight | Looker Studio |



## 📂 Dataset

- **Source:** Kaggle — YouTube Trending Videos Dataset
- **Countries:** CA, DE, FR, GB, IN, JP, KR, MX, RU, US
- **Files:** 10 CSV files + 10 JSON category files
- **Total Size:** ~539 MB
- **Raw Rows:** 375,942

Each CSV contains trending video data per country. Each JSON file maps `category_id` to a human-readable category name (e.g., `24` → `Entertainment`).


## ⚙️ Step 1 — Environment Setup
### Libraries Required

In [1]:
!pip install pandas psycopg2-binary sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 12.6 MB/s eta 0:00:00


### Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

data_path = '/content/drive/MyDrive/Colab_Notebooks/Data-Eng-Projects/Youtube-DE-Projects/Data'

os.listdir(data_path)

['CA_category_id.json',
 'CAvideos.csv',
 'DE_category_id.json',
 'DEvideos.csv',
 'FR_category_id.json',
 'GB_category_id.json',
 'FRvideos.csv',
 'IN_category_id.json',
 'GBvideos.csv',
 'JP_category_id.json',
 'INvideos.csv',
 'KR_category_id.json',
 'JPvideos.csv',
 'MX_category_id.json',
 'KRvideos.csv',
 'MXvideos.csv',
 'RU_category_id.json',
 'RUvideos.csv',
 'US_category_id.json',
 'USvideos.csv']

## 📥 Step 2 — Load Raw Data

### Load All CSV Files


In [5]:
import pandas as pd
import json
import glob

# Load all CSV files
csv_files = glob.glob(data_path +"/*.csv")

# Combine all CSV files into one dataframe
df_videos = pd.concat([pd.read_csv(f, encoding='latin-1') for f in csv_files], ignore_index=True)
print("Shape:", df_videos.shape)

Shape: (375942, 16)


In [6]:
print("\nColumns:", df_videos.columns.tolist())
print("\nFirst 5 rows:")
df_videos.head()


Columns: ['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'tags', 'views', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description']

First 5 rows:


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description
0,n1WpP7iowLc,17.14.11,Eminem - Walk On Water (Audio) ft. BeyoncÃ©,EminemVEVO,10,2017-11-10T17:00:03.000Z,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,https://i.ytimg.com/vi/n1WpP7iowLc/default.jpg,False,False,False,Eminem's new track Walk on Water ft. BeyoncÃ© ...
1,0dBIkQ4Mz1M,17.14.11,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13T17:00:00.000Z,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,13030,https://i.ytimg.com/vi/0dBIkQ4Mz1M/default.jpg,False,False,False,STill got a lot of packages. Probably will las...
2,5qpjK5DgCt4,17.14.11,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12T19:05:24.000Z,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,https://i.ytimg.com/vi/5qpjK5DgCt4/default.jpg,False,False,False,WATCH MY PREVIOUS VIDEO â¶ \n\nSUBSCRIBE âº ...
3,d380meD0W0M,17.14.11,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12T18:01:41.000Z,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,https://i.ytimg.com/vi/d380meD0W0M/default.jpg,False,False,False,I know it's been a while since we did this sho...
4,2Vv-BfVoq4g,17.14.11,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09T11:04:14.000Z,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,https://i.ytimg.com/vi/2Vv-BfVoq4g/default.jpg,False,False,False,ð§: https://ad.gt/yt-perfect\nð°: https://...


In [7]:
top_channels = df_videos['channel_title'].value_counts().head(10)
display(top_channels)

,count
channel_title,
The Late Show with Stephen Colbert,984
WWE,804
Late Night with Seth Meyers,773
VikatanTV,763
TheEllenShow,743
Jimmy Kimmel Live,707
The Tonight Show Starring Jimmy Fallon,705
PewDiePie,652
RadaanMedia,651


In [8]:
# Check data info
print("Total rows:", len(df_videos))
print("\nData types:")
print(df_videos.dtypes)
print("\nMissing values:")
print(df_videos.isnull().sum())

Total rows: 375942

Data types:
video_id                  object
trending_date             object
title                     object
channel_title             object
category_id                int64
publish_time              object
tags                      object
views                      int64
likes                      int64
dislikes                   int64
comment_count              int64
thumbnail_link            object
comments_disabled           bool
ratings_disabled            bool
video_error_or_removed      bool
description               object
dtype: object

Missing values:
video_id                      0
trending_date                 0
title                         0
channel_title                 0
category_id                   0
publish_time                  0
tags                          0
views                         0
likes                         0
dislikes                      0
comment_count                 0
thumbnail_link                0
comments_disabled        

### Load JSON Category Files

In [9]:
# Load JSON category files
import json
import glob

json_files = glob.glob(data_path + '/*.json')

category_list = []

for file in json_files:
    with open(file) as f:
        data = json.load(f)
        for item in data['items']:
            category_list.append({
                'category_id': int(item['id']),
                'category_name': item['snippet']['title']
            })

# Create dataframe and remove duplicates
df_categories = pd.DataFrame(category_list).drop_duplicates(subset='category_id')

print("Total categories:", len(df_categories))
print(df_categories)

Total categories: 32
     category_id          category_name
0              1       Film & Animation
1              2       Autos & Vehicles
2             10                  Music
3             15         Pets & Animals
4             17                 Sports
5             18           Short Movies
6             19        Travel & Events
7             20                 Gaming
8             21          Videoblogging
9             22         People & Blogs
10            23                 Comedy
11            24          Entertainment
12            25        News & Politics
13            26          Howto & Style
14            27              Education
15            28   Science & Technology
16            30                 Movies
17            31        Anime/Animation
18            32       Action/Adventure
19            33               Classics
20            34                 Comedy
21            35            Documentary
22            36                  Drama
23            37   

## 🧹 Step 3 — Clean & Transform Data

In [10]:
# Drop irrelevant columns
df_videos = df_videos.drop(columns=['description', 'thumbnail_link'])

# Fix date formats
df_videos['trending_date'] = pd.to_datetime(df_videos['trending_date'], format='%y.%d.%m')
df_videos['publish_time'] = pd.to_datetime(df_videos['publish_time'])

# Remove duplicates
df_videos = df_videos.drop_duplicates()

print("Shape after cleaning:", df_videos.shape)

Shape after cleaning: (339515, 14)


### Merge with Category Names

In [11]:
df_final = df_videos.merge(df_categories, on='category_id', how='left')

print("Shape:", df_final.shape)
print("\nSample data with category names:")
df_final[['title', 'channel_title', 'category_name', 'views', 'likes']].head(10)

Shape: (339515, 15)

Sample data with category names:


,title,channel_title,category_name,views,likes
0,Eminem - Walk On Water (Audio) ft. BeyoncÃ©,EminemVEVO,Music,17158579,787425
1,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,Comedy,1014651,127794
2,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,Comedy,3191434,146035
3,I Dare You: GOING BALD!?,nigahiga,Entertainment,2095828,132239
4,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,Music,33523622,1634130
5,Jake Paul Says Alissa Violet CHEATED with LOGA...,DramaAlert,News & Politics,1309699,103755
6,Vanoss Superhero School - New Students,VanossGaming,Comedy,2987945,187464
7,WE WANT TO TALK ABOUT OUR MARRIAGE,CaseyNeistat,People & Blogs,748374,57534
8,THE LOGANG MADE HISTORY. LOL. AGAIN.,Logan Paul Vlogs,Entertainment,4477587,292837
9,Finally Sheldon is winning an argument about t...,Sheikh Musa,People & Blogs,505161,4135


## 🗄️ Step 4 — Load into Neon PostgreSQL

### Connect to Database

In [13]:
from sqlalchemy import create_engine

# Neon connection
host = "ep-lively-morning-aly8r2sv.c-3.eu-central-1.aws.neon.tech"
database = "neondb"
user = "neondb_owner"
password = "npg_Sj2ind7Ufxey"

connection_string = f"postgresql+psycopg2://{user}:{password}@{host}/{database}?sslmode=require"

engine = create_engine(connection_string)

# Test connection
with engine.connect() as conn:
    print("✅ Connected to Neon PostgreSQL successfully!")

✅ Connected to Neon PostgreSQL successfully!


### Load Data

In [14]:
# Load data into Neon PostgreSQL
print("Loading data... this may take a few minutes...")

df_final.to_sql(
    name='youtube_videos',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=1000
)

print(f"✅ Done! {len(df_final)} rows loaded into Neon database!")

Loading data... this may take a few minutes...
✅ Done! 339515 rows loaded into Neon database!


### Verify

In [15]:
import pandas as pd

# Verify data in database
df_check = pd.read_sql("SELECT COUNT(*) as total_rows FROM youtube_videos", engine)
print("Total rows in database:", df_check['total_rows'][0])

# Preview data from database
df_preview = pd.read_sql("SELECT * FROM youtube_videos LIMIT 5", engine)
df_preview.head()


Total rows in database: 339515


,video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,comments_disabled,ratings_disabled,video_error_or_removed,category_name
0,n1WpP7iowLc,2017-11-14,Eminem - Walk On Water (Audio) ft. BeyoncÃ©,EminemVEVO,10,2017-11-10 17:00:03+00:00,"Eminem|""Walk""|""On""|""Water""|""Aftermath/Shady/In...",17158579,787425,43420,125882,False,False,False,Music
1,0dBIkQ4Mz1M,2017-11-14,PLUSH - Bad Unboxing Fan Mail,iDubbbzTV,23,2017-11-13 17:00:00+00:00,"plush|""bad unboxing""|""unboxing""|""fan mail""|""id...",1014651,127794,1688,13030,False,False,False,Comedy
2,5qpjK5DgCt4,2017-11-14,"Racist Superman | Rudy Mancuso, King Bach & Le...",Rudy Mancuso,23,2017-11-12 19:05:24+00:00,"racist superman|""rudy""|""mancuso""|""king""|""bach""...",3191434,146035,5339,8181,False,False,False,Comedy
3,d380meD0W0M,2017-11-14,I Dare You: GOING BALD!?,nigahiga,24,2017-11-12 18:01:41+00:00,"ryan|""higa""|""higatv""|""nigahiga""|""i dare you""|""...",2095828,132239,1989,17518,False,False,False,Entertainment
4,2Vv-BfVoq4g,2017-11-14,Ed Sheeran - Perfect (Official Music Video),Ed Sheeran,10,2017-11-09 11:04:14+00:00,"edsheeran|""ed sheeran""|""acoustic""|""live""|""cove...",33523622,1634130,21082,85067,False,False,False,Music


## 🔍 Step 5 — Analyze Data with SQL

### Top 10 Most Viewed Videos (Unique)

In [16]:
# Top 10 most viewed videos
df_top_views = pd.read_sql("""
    SELECT title, channel_title, category_name, views
    FROM youtube_videos
    ORDER BY views DESC
    LIMIT 10
""",engine)
df_top_views



,title,channel_title,category_name,views
0,Nicky Jam x J. Balvin - X (EQUIS) | Video Ofic...,NickyJamTV,Music,424538912
1,Nicky Jam x J. Balvin - X (EQUIS) | Video Ofic...,NickyJamTV,Music,413586699
2,Nicky Jam x J. Balvin - X (EQUIS) | Video Ofic...,NickyJamTV,Music,402650804
3,Nicky Jam x J. Balvin - X (EQUIS) | Video Ofic...,NickyJamTV,Music,392036878
4,Nicky Jam x J. Balvin - X (EQUIS) | Video Ofic...,NickyJamTV,Music,382401497
5,Nicky Jam x J. Balvin - X (EQUIS) | Video Ofic...,NickyJamTV,Music,372399338
6,Nicky Jam x J. Balvin - X (EQUIS) | Video Ofic...,NickyJamTV,Music,362111555
7,Nicky Jam x J. Balvin - X (EQUIS) | Video Ofic...,NickyJamTV,Music,349987176
8,Nicky Jam x J. Balvin - X (EQUIS) | Video Ofic...,NickyJamTV,Music,339629489
9,"Te Bote Remix - Casper, Nio GarcÃ­a, Darell, N...",Flow La Movie,Music,337621571


In [17]:
# Top 10 videos by unique views (no duplicates)
df_top_unique = pd.read_sql("""
    SELECT title, channel_title, category_name, MAX(views) as max_views
    FROM youtube_videos
    GROUP BY title, channel_title, category_name
    ORDER BY max_views DESC
    LIMIT 10
""", engine)

print("🏆 Top 10 Most Viewed Videos:")
print(df_top_unique.to_string(index=False))

🏆 Top 10 Most Viewed Videos:
                                                                                   title       channel_title category_name  max_views
              Nicky Jam x J. Balvin - X (EQUIS) | Video Oficial | Prod. Afro Bros & Jeon          NickyJamTV         Music  424538912
Te Bote Remix - Casper, Nio GarcÃ­a, Darell, Nicky Jam, Bad Bunny, Ozuna | Video Oficial       Flow La Movie         Music  337621571
                                                    Bad Bunny - Amorfoda | Video Oficial           Bad Bunny         Music  328860380
                                                Ozuna x Romeo Santos - El Farsante Remix               Ozuna         Music  288811992
                                     Childish Gambino - This Is America (Official Video) ChildishGambinoVEVO         Music  259721696
                                                                    Drake - Godâs Plan           DrakeVEVO         Music  258164991
                                 

### Top Categories by Total Views

In [18]:
# Top 10 categories by total views
df_categories_views = pd.read_sql("""
    SELECT category_name,
           COUNT(*) as total_videos,
           SUM(views) as total_views,
           ROUND(AVG(likes)) as avg_likes
    FROM youtube_videos
    GROUP BY category_name
    ORDER BY total_views DESC
    LIMIT 10
""", engine)

print("\n📊 Top 10 Categories by Views:")
print(df_categories_views.to_string(index=False))


📊 Top 10 Categories by Views:
       category_name  total_videos  total_views  avg_likes
               Music         36990 2.203896e+11   158633.0
       Entertainment         97361 8.515823e+10    24422.0
    Film & Animation         18329 2.039738e+10    23875.0
      People & Blogs         50164 1.917844e+10    11629.0
              Comedy         24024 1.834101e+10    38771.0
              Sports         21406 1.503582e+10    15428.0
     News & Politics         34598 8.816664e+09     4104.0
       Howto & Style         17548 8.269928e+09    17480.0
Science & Technology          7350 6.986113e+09    27634.0
              Gaming         10433 6.216419e+09    24132.0


### Engagement Rate by Category

In [19]:
# Engagement analysis - likes vs views ratio per category
df_engagement = pd.read_sql("""
    SELECT
        category_name,
        COUNT(DISTINCT title) as unique_videos,
        ROUND(AVG(views)) as avg_views,
        ROUND(AVG(likes)) as avg_likes,
        ROUND(AVG(likes) * 100.0 / NULLIF(AVG(views), 0), 2) as like_rate_percent
    FROM youtube_videos
    WHERE category_name IS NOT NULL
    GROUP BY category_name
    ORDER BY like_rate_percent DESC
    LIMIT 10
""", engine)

print("❤️ Categories with Highest Engagement (Likes/Views %):")
print(df_engagement.to_string(index=False))

❤️ Categories with Highest Engagement (Likes/Views %):
        category_name  unique_videos  avg_views  avg_likes  like_rate_percent
Nonprofits & Activism           2022   400477.0    30663.0               7.66
               Comedy          10866   763445.0    38771.0               5.08
            Education           4135   335999.0    14204.0               4.23
               Gaming           5660   595842.0    24132.0               4.05
        Howto & Style           9782   471275.0    17480.0               3.71
       People & Blogs          33623   382315.0    11629.0               3.04
 Science & Technology           3516   950492.0    27634.0               2.91
     Autos & Vehicles           3444   329461.0     9473.0               2.88
       Pets & Animals           2364   395920.0    11205.0               2.83
        Entertainment          53564   874665.0    24422.0               2.79


### Trending Days of the Week

In [21]:
# Trending days analysis - which day of week videos trend most
df_trending_days = pd.read_sql("""
    SELECT
        TO_CHAR(trending_date, 'Day') as day_of_week,
        COUNT(*) as total_trending_videos,
        ROUND(AVG(views)) as avg_views
    FROM youtube_videos
    WHERE category_name IS NOT NULL
    GROUP BY TO_CHAR(trending_date, 'Day')
    ORDER BY total_trending_videos DESC
""", engine)

print("📅 Which day videos trend the most:")
print(df_trending_days.to_string(index=False))

📅 Which day videos trend the most:
day_of_week  total_trending_videos  avg_views
  Saturday                   50154  1232367.0
  Tuesday                    50057  1189413.0
  Monday                     48162  1232106.0
  Sunday                     48147  1248657.0
  Friday                     48061  1192393.0
  Wednesday                  47649  1250236.0
  Thursday                   47285  1244785.0


In [22]:
# Export cleaned data to CSV
df_final.to_csv('/content/drive/MyDrive/Colab_Notebooks/Data-Eng-Projects/Youtube-DE-Projects/youtube_cleaned.csv', index=False)

print("✅ File saved to Google Drive!")

✅ File saved to Google Drive!


## 📊 Step 6 — Dashboard (Looker Studio)

### Connecting Looker Studio to Neon PostgreSQL

1. Go to [lookerstudio.google.com](https://lookerstudio.google.com)
2. Click **Create** → **Report**
3. Search for **PostgreSQL** connector
4. Download the SSL certificate: [isrgrootx1.pem](https://letsencrypt.org/certs/isrgrootx1.pem)
5. Fill in your Neon connection details:


6. Use **Custom Query** to load data:

```sql
SELECT title, channel_title, category_name,
       views, likes, dislikes, comment_count,
       trending_date, publish_time
FROM youtube_videos
WHERE category_name IS NOT NULL
```



## 💡 Key Insights

- **Music** and **Entertainment** are the most viewed categories globally
- **Nonprofits & Activism** and **Comedy** have the highest engagement rates (likes per view)
- Videos trend most on **Saturday** and **Tuesday**
- **NickyJamTV** is the most viewed channel in the dataset with over **9 billion views**


## 📁 Project Structure

```
Youtube-DE-Projects/
│
├── Data/
│   ├── CAvideos.csv
│   ├── DEvideos.csv
│   ├── FRvideos.csv
│   ├── GBvideos.csv
│   ├── INvideos.csv
│   ├── JPvideos.csv
│   ├── KRvideos.csv
│   ├── MXvideos.csv
│   ├── RUvideos.csv
│   ├── USvideos.csv
│   ├── CA_category_id.json
│   └── ... (10 JSON files)
│
├── youtube_de_pipeline.ipynb   ← Main Colab Notebook
└── youtube_cleaned.csv         ← Cleaned output
```


## 🙏 Credits

- Original project concept by **[Darshil Parmar](https://www.youtube.com/@DarshilParmar)**
- Dataset from **[Kaggle — YouTube Trending Videos](https://www.kaggle.com/datasets/datasnaek/youtube-new)**
- Free infrastructure: **Google Colab**, **Neon.tech**, **Looker Studio**
